# Задание по программированию: Рекомендательные системы

Библиотеки

In [1]:
import pandas as pd
import numpy as np

%pylab inline
import pandas as pd
from scipy import stats
import statsmodels
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
import itertools

warnings.simplefilter('ignore')

# fix cyrillic fonts
from matplotlib import rc
font = {'family': 'Verdana','weight': 'normal'}
rc('font', **font)


Populating the interactive namespace from numpy and matplotlib


Считаем данные

In [2]:
def read_sessions(file_name):
    sessions = []
    with open(file_name, 'r') as f:
        for l in f.readlines():
            parts = str.split(l.strip(), ';')
            sessions.append(([int(_) for _ in str.split(parts[0], ',')], 
                      [int(_) for _ in str.split(parts[1], ',')] if len(parts[1]) > 0 else []))
    return sessions


train = read_sessions ('coursera_sessions_train.txt')
test = read_sessions ('coursera_sessions_test.txt')

In [3]:
train[:5]

[([0, 1, 2, 3, 4, 5], []),
 ([9, 10, 11, 9, 11, 12, 9, 11], []),
 ([16, 17, 18, 19, 20, 21], []),
 ([24, 25, 26, 27, 24], []),
 ([34, 35, 36, 34, 37, 35, 36, 37, 38, 39, 38, 39], [])]

In [4]:
test[:5]

[([6, 7, 8], []),
 ([13, 14, 15], []),
 ([22, 23], []),
 ([28, 29, 30, 31, 32, 33], []),
 ([40, 41], [])]

In [5]:
from collections import Counter, defaultdict

def calc_model(sessions):
    views = defaultdict(int)
    purchases = defaultdict(int)
    for sess in sessions:
        for k, v in Counter(sess[0]).items():
            views[k] += v
        if len(sess[1]):
            for k, v in Counter(sess[1]).items():
                purchases[k] += v

    return views, purchases


In [6]:
train_views, train_purchases = calc_model(train)
test_views, test_purchases = calc_model(test)

In [7]:
train_purchases

defaultdict(int,
            {49152: 1,
             5: 1,
             6: 2,
             7: 2,
             8: 3,
             35501: 1,
             49168: 1,
             17: 2,
             81943: 1,
             16409: 1,
             28: 3,
             29: 1,
             30: 1,
             49183: 1,
             32: 1,
             16421: 1,
             38: 1,
             16424: 1,
             42: 1,
             44: 1,
             35506: 1,
             46: 1,
             49200: 1,
             49160: 1,
             50: 1,
             16441: 1,
             60: 1,
             63: 1,
             16448: 1,
             16450: 1,
             67: 1,
             71: 4,
             72: 1,
             73: 11,
             80: 1,
             32850: 1,
             83: 1,
             85: 1,
             86: 2,
             32855: 1,
             62820: 1,
             93: 1,
             16478: 1,
             16479: 2,
             99: 1,
             16490: 1,
      

In [8]:
train_views_list = np.array(list([k,v] for k,v in train_views.items()))
train_purchases_list = np.array(list([k,v] for k,v in train_purchases.items()))

train_views_sorted = train_views_list[train_views_list[:,1].argsort(kind = 'mergesort')][::-1]
train_purchases_sorted = train_purchases_list[train_purchases_list[:,1].argsort(kind = 'mergesort')][::-1]

print(train_views_sorted[:5], '\n\n', train_views_sorted[-5:])
print(train_purchases_sorted[:5], '\n\n', train_purchases_sorted[-5:])

[[ 73 677]
 [158 641]
 [204 396]
 [262 387]
 [162 318]] 

 [[59  1]
 [58  1]
 [56  1]
 [53  1]
 [49  1]]
[[ 158   14]
 [ 204   12]
 [3324   11]
 [  73   11]
 [5569   10]] 

 [[57354     1]
 [ 9558     1]
 [47787     1]
 [53738     1]
 [49152     1]]


In [20]:
import operator
def recommend_by(sessions, recommendations_model):
    #   теперь надо показывать не последние просмотренные пользователем товары, 
    # а те товары из просмотренных, которые он наиболее вероятно купит.
    metrics = np.zeros(4) 
    count = 0
    for sess in sessions:
        # skip empty sessions
        if not sess[1]:
            continue

        count += 1
        views = pd.unique(sess[0])
        purchases = sess[1]
        
        popularities = [recommendations_model[id] for id in views]
        
        # works for 1 and 2
        # recommended = views[np.argsort(popularities, kind = 'mergesort')][::-1]
        
        popularities_2 = [(id, recommendations_model[id]) for id in views]
        recommended = list(map(operator.itemgetter(0), 
                               sorted(popularities_2, key=operator.itemgetter(1), reverse=True)))
        
        # print(purchases, recommended)
        # break
        
        # AverageRecall@1, AveragePrecision@1, AverageRecall@5, AveragePrecision@5
        precision1 = 1. if recommended[0] in purchases else 0.
        recall1 = 1./len(purchases) if recommended[0] in purchases else 0.
        
        recommended_count = min(5, len(views))
        purchases_intercept = len(np.intersect1d(purchases, recommended[:5]))
        
        precision5 = purchases_intercept / 5.
        recall5 = purchases_intercept / len(purchases)
                
        metrics[0] += recall1
        metrics[1] += precision1
        metrics[2] += recall5
        metrics[3] += precision5
        
    return np.round(metrics / count, 2)

def write_answer(filename, metrics):
    metrics_str = ' '.join(([str(round(_, 2)) for _ in metrics]))
    print(filename, ':', metrics_str)
    with open(filename, 'w') as f:
        f.write(metrics_str)   
        

In [21]:
write_answer('recommend_by_views_train.txt', recommend_by(train, train_views))

recommend_by_views_train.txt : 0.44 0.51 0.82 0.21


In [22]:
write_answer('recommend_by_views_test.txt', recommend_by(test, train_views))

recommend_by_views_test.txt : 0.42 0.48 0.8 0.2


In [23]:
res = recommend_by(train, train_purchases)    
invalid=[0.66, 0.78, 0.92, 0.25]
print(res == invalid)
write_answer('recommend_by_purchases_train.txt', recommend_by(train, train_purchases))             


[False False False  True]
recommend_by_purchases_train.txt : 0.69 0.8 0.93 0.25


In [19]:
res = recommend_by(test, train_purchases)    
invalid=[0.4, 0.46, 0.76, 0.2]
print(res == invalid)
write_answer('recommend_by_purchases_test.txt', res)


[False False False False]
recommend_by_purchases_test.txt : 0.46 0.53 0.82 0.21


In [362]:
# test
views_t = ['59', '60', '61', '62', '60', '63', '64', '65', '66', '61', '67', '68', '67']
purchases_t = ['67','60', '63']
recommend_by([[views_t, purchases_t]], Counter(views_t))


array([ 0.33,  1.  ,  0.67,  0.4 ])